# 21 · Is the agent learning anything? (E21, train + validation only)

Every RL result so far (E5-E7, E16-E17, E20) failed to beat buy and hold reliably. That can mean three different
things, and each needs a different fix:

1. **it cannot learn at all**: even on its own training data the greedy policy is no better than before training
   (the reward, state, decision interval or costs make the problem unlearnable);
2. **it learns the training data only**: good in-sample, nothing that carries over (overfitting / changing market);
3. **it learns, but not from the factors**: a direction bet that ignores the inputs.

Same agent as E16/E20 (Double DQN, state A, E16 hyperparameters, 0.24 % round-trip cost, no entry gate),
1M steps, 2 seeds per variant. The greedy policy is scored on **train and validation** before training and every
100k steps.

| test | factors | question |
|---|---|---|
| A | real (train cross-fitted from 20a, validation E15) | does in-sample performance rise with training, and does validation follow? |
| B | real values, whole days moved to random other days | no information left: if A ≈ B, the agent is not using the factors |
| C | f5 (the 300-bar factor) replaced by a noisy copy of the **future** 300-bar return at IC 0.05 / 0.1 / 0.2 / 0.4 | the weakest signal this setup can turn into profit (honest factors sit near IC 0.1) |
| D | (analysis) | does the agent's long-vs-short preference track the predictive factors? Also for the 5 saved E16 agents |

**Test C is a deliberate leak**, used only as a positive control of the learning setup, never as a trading
result. Each variant also gets the E19 benchmark (best threshold rule on train, traded on validation). No
checkpoint is selected for reporting: the curves show every checkpoint. **Test and stress_bear are not loaded.**

T4 GPU, about 20-25 min per run, 12 runs (about 4-5 h); runs resume after a disconnect. Writes
`thesis_rl/initiation/e21_learnability/` and `report/e21_*`.

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/14-initiation"
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available(): print('WARNING: no GPU - Runtime > Change runtime type > T4 GPU')

In [ ]:
seeding.record_environment(L.report / "environment", "21_learnability_check")

In [ ]:
import json, time, zipfile, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from rltrade import factor_rule as FR, learnability as LB, walkforward as WF
from rltrade.env import EnvConfig, Market
from rltrade.runner import load_saved

INIT = L.root / "initiation"
OUT, REP = INIT / "e21_learnability", INIT / "report"
for p in (OUT, REP): p.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/initiation_prepared") if "google.colab" in sys.modules else INIT / "prepared"
ZIP = INIT / "initiation_prepared.zip"
PREP = LOCAL / "fixed_plus"
NEEDED = [f"fixed_plus/market_{s}.npz" for s in ("train", "val")] + ["fixed_plus/meta.json"]
if not all((LOCAL / n).exists() for n in NEEDED):
    assert ZIP.exists(), f"upload initiation_prepared.zip to {ZIP}"
    with zipfile.ZipFile(ZIP) as z: z.extractall(LOCAL, members=NEEDED)   # test / stress_bear stay zipped
    print("unzipped", NEEDED, "->", LOCAL)
F = INIT / "e20" / "crossfit_mean3"
assert (F / "crossfit.json").exists(), "run notebook 20a first (cross-fitted factors)"

hp = json.loads((INIT / "agents" / "tuning" / "best_hp.json").read_text())       # E16's tuned Double DQN
AGENT_HP, ENV_OVER = hp["agent_hp"], hp["env"]
AGENT_HP["net_dims"] = tuple(AGENT_HP["net_dims"])
CFG = EnvConfig(state="A", **ENV_OVER)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
SEEDS, TOTAL_STEPS, EVAL_EVERY, MIN_TRADES = (0, 1), 1_000_000, 100_000, 5
TRAIN_CHUNKS, VAL_CHUNKS = 32, 16
RHOS = (0.4, 0.2, 0.1, 0.05)        # planted-signal correlations, strongest first

VARIANTS = {"real": F}
VARIANTS["shuffled"], info = LB.write_variant(PREP, F, LOCAL / "e21_factors" / "shuffled", "shuffled", seed=0)
VINFO = {"shuffled": info}
for rho in RHOS:
    name = f"planted_{rho:g}"
    VARIANTS[name], VINFO[name] = LB.write_variant(PREP, F, LOCAL / "e21_factors" / name, "planted", rho=rho, seed=0)
PRICE = {s: np.load(PREP / f"market_{s}.npz")["midpoint"].astype(np.float64) for s in ("train", "val")}
FACT = {(v, s): np.load(Path(d) / f"factors_{s}.npy") for v, d in VARIANTS.items() for s in ("train", "val")}
ic_rows = []
for v in VARIANTS:
    r = {"variant": v}
    for s in ("train", "val"):
        r[f"f{LB.PLANT_FACTOR} IC {s}"] = FR.ic(FACT[(v, s)][:, LB.PLANT_FACTOR], FR.forward_returns(PRICE[s], LB.PLANT_H), LB.PLANT_H)[0]
    ic_rows.append(r)
ic_df = pd.DataFrame(ic_rows)
print(f"factor f{LB.PLANT_FACTOR} vs the {LB.PLANT_H}-bar forward return (Spearman, non-overlapping):")
print(ic_df.round(3).to_string(index=False))
print("agent:", AGENT_HP, ENV_OVER, "| device", DEV)

## 1 · References: buy and hold, and the best threshold rule on train (per variant)

In [ ]:
MK = {(v, s): Market(PREP, d, s, CFG, DEV) for v, d in VARIANTS.items() for s in ("train", "val")}
BH = {s: LB.reference_scores(MK[("real", s)], CFG, c) for s, c in (("train", TRAIN_CHUNKS), ("val", VAL_CHUNKS))}
BH_EXACT = {s: LB.reference_scores(MK[("real", s)], CFG, 1) for s in ("train", "val")}
ref_rows = []
for v in VARIANTS:
    grid = FR.make_grid(FR.signals(FACT[(v, "train")]))
    if v.startswith("planted"):                  # the planted signal is f5; its rules are enough (and faster)
        grid = [g for g in grid if g["j"] == LB.PLANT_FACTOR]
    tr = FR.run_rules(grid, MK[(v, "train")], CFG)
    va = FR.run_rules(grid, MK[(v, "val")], CFG)
    i = WF.select_rule(tr, MIN_TRADES)
    g = grid[i]
    ref_rows.append({"variant": v, "rules": len(grid), "best rule on train": f"{g['signal']} q{g['q']} {g['mode']} hold>={g['min_hold']}",
                     "train Sharpe": tr["sharpe"][i], "train return %": tr["cum_return_pct"][i],
                     "val Sharpe": va["sharpe"][i], "val return %": va["cum_return_pct"][i], "val trades": int(va["num_trades"][i])})
ref_df = pd.DataFrame(ref_rows)
ref_df.to_csv(OUT / "rule_reference.csv", index=False)
print(f"buy and hold (exact): train Sharpe {BH_EXACT['train']['sharpe']:+.2f} ({BH_EXACT['train']['cum_return_pct']:+.1f} %), "
      f"val Sharpe {BH_EXACT['val']['sharpe']:+.2f} ({BH_EXACT['val']['cum_return_pct']:+.1f} %); cash 0")
print(f"buy and hold as scored in the curves ({TRAIN_CHUNKS} / {VAL_CHUNKS} chunks): train {BH['train']['sharpe']:+.2f}, val {BH['val']['sharpe']:+.2f}")
print(ref_df.round(3).to_string(index=False))

## 2 · Train the agent on each variant (about 20-25 min per run; resumes)

In [ ]:
def run_name(v, seed):
    return f"{v}_s{seed}"

t0 = time.time()
for seed in SEEDS:                      # all variants for seed 0 first, so a partial run is already informative
    for v, d in VARIANTS.items():
        LB.train_run(run_name(v, seed), PREP, d, OUT, seed, AGENT_HP, ENV_OVER, TOTAL_STEPS, EVAL_EVERY,
                     device=DEV, train_chunks=TRAIN_CHUNKS, val_chunks=VAL_CHUNKS)
print(f"training done ({(time.time() - t0) / 60:.0f} min this session)")

## 3 · Results: does performance rise with training, on train and on validation?

In [ ]:
RES = {(v, s): json.loads((OUT / "runs" / run_name(v, s) / "result.json").read_text()) for v in VARIANTS for s in SEEDS}
run_rows = []
for (v, s), r in RES.items():
    c = pd.DataFrame(r["curve"])
    ft, fv = r["final"]["train"], r["final"]["val"]
    run_rows.append({"variant": v, "seed": s,
                     "train Sharpe @0": c.train_sharpe.iloc[0], "train Sharpe best": c.train_sharpe.max(),
                     "train Sharpe final": ft["sharpe"], "train return % final": ft["cum_return_pct"],
                     "train timing %": ft["attr_timing_pct"], "train gross %": ft["gross_pct"],
                     "val Sharpe @0": c.val_sharpe.iloc[0], "val Sharpe best": c.val_sharpe.max(),
                     "val Sharpe final": fv["sharpe"], "val return % final": fv["cum_return_pct"],
                     "val timing %": fv["attr_timing_pct"], "val trades": fv["num_trades"],
                     "val in market %": fv["exposure_pct"], "val costs %": fv["costs_pct"],
                     "val daily timing IC": fv["daily_ic"]})
runs = pd.DataFrame(run_rows)
runs.to_csv(OUT / "runs_summary.csv", index=False)
num = runs.drop(columns=["seed"]).groupby("variant", sort=False).mean()
var_df = num.join(ic_df.set_index("variant")).join(ref_df.set_index("variant")[["val Sharpe"]].rename(columns={"val Sharpe": "rule val Sharpe"}))
print("every run (final = the policy after 1M steps, one exact pass; timing = what position changes added beyond the average exposure):")
print(runs.round(2).to_string(index=False))
print("\nmean over seeds, per variant:")
print(var_df.round(2).T.to_string())

In [ ]:
COL = {v: f"C{i}" for i, v in enumerate(VARIANTS)}
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
for ax, split in zip(axes, ("train", "val")):
    for v in VARIANTS:
        for s in SEEDS:
            c = pd.DataFrame(RES[(v, s)]["curve"])
            ax.plot(c.steps / 1e6, c[f"{split}_sharpe"], color=COL[v], lw=1.4 if s == SEEDS[0] else 0.8,
                    alpha=0.9 if s == SEEDS[0] else 0.5, label=v if s == SEEDS[0] else None)
    ax.axhline(BH[split]["sharpe"], color="black", ls="--", lw=1, label="buy and hold")
    ax.axhline(0, color="grey", lw=0.6)
    ax.set_title(f"{split}: greedy policy Sharpe during training (net of costs)", fontsize=10)
    ax.set_xlabel("training steps (millions)"); ax.grid(alpha=0.3)
axes[0].set_ylabel("Sharpe"); axes[1].legend(fontsize=8, ncol=2)
fig.tight_layout(); fig.savefig(REP / "e21_learning_curves.png", dpi=130); plt.show()

pl = [v for v in VARIANTS if v.startswith("planted")]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, split in zip(axes, ("train", "val")):
    x = ic_df.set_index("variant").loc[pl, f"f{LB.PLANT_FACTOR} IC {split}"]
    for s in SEEDS:
        ax.scatter(x, [RES[(v, s)]["final"][split]["sharpe"] for v in pl], color="C2", s=36, label="agent (each seed)" if s == SEEDS[0] else None, zorder=3)
    if split == "val":
        ax.plot(x, ref_df.set_index("variant").loc[pl, "val Sharpe"], color="C9", marker="s", lw=1.2, label="best f5 rule on train")
    else:
        ax.plot(x, ref_df.set_index("variant").loc[pl, "train Sharpe"], color="C9", marker="s", lw=1.2, label="best f5 rule on train")
    ax.axhline(BH_EXACT[split]["sharpe"], color="black", ls="--", lw=1, label="buy and hold")
    for v, ls in (("real", ":"), ("shuffled", "-.")):
        ax.axhline(np.mean([RES[(v, s)]["final"][split]["sharpe"] for s in SEEDS]), color=COL[v], ls=ls, lw=1, label=f"agent, {v} factors")
    ax.axhline(0, color="grey", lw=0.6)
    ax.set_xlabel(f"realised IC of the planted f5 ({split})"); ax.set_title(f"{split}: final policy vs signal strength", fontsize=10); ax.grid(alpha=0.3)
axes[0].set_ylabel("Sharpe (exact pass)"); axes[1].legend(fontsize=7)
fig.tight_layout(); fig.savefig(REP / "e21_signal_strength.png", dpi=130); plt.show()

## 4 · Test D: what drives the decision? (Q(long) - Q(short) from a flat position)

In [ ]:
pref_rows = []
for (v, s) in RES:
    agent, cfg = LB.load_agent(OUT / "runs" / run_name(v, s), DEV)
    for split in ("train", "val"):
        p, a = LB.q_preference(agent, MK[(v, split)])
        pref_rows.append({"agent": f"E21 {v}", "seed": s, "split": split,
                          **LB.preference_table(p, a, FACT[(v, split)], PRICE[split])})
E16 = INIT / "agents" / "models" / "16_agents"
for rd in sorted(E16.glob("ddqn_A_*_s*")) if E16.exists() else []:
    _, agent, cfg, conf, _ = load_saved(rd, DEV, use_fallback=False)
    fd = Path(conf["factor_dir"])
    if not fd.exists():
        fd = INIT / fd.relative_to(fd.parents[1]) if len(fd.parents) > 1 else fd
    for split in ("train", "val"):
        m = Market(PREP, fd, split, cfg, DEV)
        p, a = LB.q_preference(agent, m)
        pref_rows.append({"agent": "E16 (E15 factors)", "seed": conf["spec"]["seed"], "split": split,
                          **LB.preference_table(p, a, np.load(fd / f"factors_{split}.npy"), PRICE[split])})
pref = pd.DataFrame(pref_rows)
pref.to_csv(OUT / "q_preference.csv", index=False)
cols = ["agent", "split"] + [f"corr_f{j}" for j in range(8)] + [f"ic_{h}" for h in (60, LB.PLANT_H, 900)] + ["from_flat_long_pct", "from_flat_short_pct", "from_flat_stay_pct"]
pref_mean = pref[cols].groupby(["agent", "split"], sort=False).mean()
print("correlation of the long-vs-short preference with each factor, and its IC vs the forward return (mean over seeds):")
print(pref_mean.round(3).to_string())

In [ ]:
(REP / "e21_learnability.md").write_text(
    "# E21: is the agent learning anything? (train + validation only)\n\n"
    f"Double DQN, state A, E16 hyperparameters, {len(SEEDS)} seeds x {TOTAL_STEPS:,} steps per variant; greedy policy scored "
    f"on train ({TRAIN_CHUNKS} chunks) and validation ({VAL_CHUNKS} chunks) at step 0 and every {EVAL_EVERY:,} steps; final = one exact pass. "
    f"Factors: {F}. Planted variants replace f{LB.PLANT_FACTOR} with a noisy copy of the future {LB.PLANT_H}-bar return (a deliberate leak, "
    "positive control only). Test and stress_bear not loaded.\n\n"
    f"Buy and hold (exact): train Sharpe {BH_EXACT['train']['sharpe']:+.2f} ({BH_EXACT['train']['cum_return_pct']:+.1f} %), "
    f"validation {BH_EXACT['val']['sharpe']:+.2f} ({BH_EXACT['val']['cum_return_pct']:+.1f} %); cash 0.\n\n"
    "## Signal strength of f5 per variant\n\n" + ic_df.round(3).to_markdown(index=False)
    + "\n\n## Rule reference (best rule on train)\n\n" + ref_df.round(3).to_markdown(index=False)
    + "\n\n## Per variant (mean over seeds)\n\n" + var_df.round(3).reset_index().to_markdown(index=False)
    + "\n\n## Every run\n\n" + runs.round(3).to_markdown(index=False)
    + "\n\n## Test D: long-vs-short preference vs factors and forward returns\n\n" + pref_mean.round(3).reset_index().to_markdown(index=False)
    + "\n\nFigures: e21_learning_curves.png, e21_signal_strength.png\n", encoding="utf-8")
print("wrote", REP / "e21_learnability.md", "|", OUT)

In [ ]:
AUTO_DISCONNECT = True               # release the GPU once the results are on Drive
if AUTO_DISCONNECT and "google.colab" in sys.modules:
    from google.colab import runtime
    runtime.unassign()